# PayJoy FPD_15 Prediction — Best Model Pipeline (v4)

**Purpose:** Reproducible, end-to-end pipeline that trains the winning model
on all available labelled data (Jan–Nov 2025) and generates `submission.csv`
for the December 2025 test set. This notebook contains **zero experimentation
code** — all tuning was done in `model_experiments.ipynb` and the winning
configuration is hardcoded here.

**Architecture:** Neural Network (FPDNet MLP) with v4 features — smoothed
target encoding, CITY, MODEL, days_to_first_due. Nov AUC: 0.5661 (best of
XGBoost 0.5565, LightGBM 0.5512, NN 0.5661, ensemble 0.5656).

**Evaluation metric:** AUC-ROC (competition standard).

**How to use:** Run all cells top-to-bottom. The output is `submission.csv`
in the working directory, ready for Kaggle upload.

In [ ]:
import numpy as np
np.random.seed(42)
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

## Step 1 — Load Raw Data

Three CSV files are required in the working directory:

| File | Description |
|---|---|
| `Orders.csv` | One row per phone-finance order (Jan–Dec 2025). Jan–Nov rows have a known `FPD_15` label; Dec rows are `NaN` (prediction targets). |
| `Payment_History.csv` | Monthly payment snapshots for pre-December orders (~5.7M rows). |
| `Test_OrderIDs.csv` | The specific December order IDs we must submit predictions for. |

In [ ]:
orders = pd.read_csv('Orders.csv', low_memory=False)
payments = pd.read_csv('Payment_History.csv')
test_ids = pd.read_csv('Test_OrderIDs.csv')

print(f'Orders:   {orders.shape}')
print(f'Payments: {payments.shape}')
print(f'Test IDs: {len(test_ids)}')

## Step 2 — Data Preparation

The preparation pipeline performs six operations in sequence:

1. **Payment aggregation** — collapse monthly payment snapshots into one row
   per order (max cumulative paid, worst delinquency, mean delinquency,
   months of history).
2. **Feature engineering** — derive financial ratios (down-payment ratio,
   finance ratio), temporal features (hour, day-of-week, month), merchant
   tenure, entity-level historical FPD rates, state-level FPD rate
   (target-encoded), and entity order volume counts.
3. **Train/test split** — rows with non-null `FPD_15` (Jan–Nov) become the
   training set; rows with null `FPD_15` (Dec) become the test set.
4. **Imputation** — fill missing numerics with training-set medians, missing
   categoricals with training-set modes. December orders have no payment
   history, so their payment columns are imputed here.
5. **One-hot encoding** — expand low-cardinality categoricals (COUNTRY,
   LOCK_PRODUCT, MANUFACTURER, LOCK_NAME, CURRENCY) and align columns
   between train and test.
6. **Scaling** — z-score standardization fitted on training data only.

In [ ]:
def _aggregate_payments(payments_df):
    """Collapse monthly payment snapshots into one row per order.

    Returns a DataFrame keyed on FINANCEORDERID with columns:
      pay_max_cumpaid, pay_max_overdue, pay_mean_overdue, pay_num_snapshots.
    December orders have no payment history — they get NaN here and are
    imputed downstream.
    """
    agg = payments_df.groupby('FINANCEORDERID').agg(
        pay_max_cumpaid=('TOTAL_CUMPAID', 'max'),
        pay_max_overdue=('DAYSOVERDUE', 'max'),
        pay_mean_overdue=('DAYSOVERDUE', 'mean'),
        pay_num_snapshots=('CALENDAR_DATE', 'nunique'),
    ).reset_index()
    return agg


def _engineer_features(df, rate_mask=None):
    """v4: Smoothed target encoding, CITY, MODEL, days_to_first_due."""
    df['down_payment_ratio'] = (
        df['DOWN_PAYMENT_AMOUNT'] / df['PURCHASE_AMOUNT'].replace(0, np.nan)
    )
    df['finance_ratio'] = (
        df['FINANCE_AMOUNT'] / df['PURCHASE_AMOUNT'].replace(0, np.nan)
    )

    tx = pd.to_datetime(df['TRANSACTIONTIME'], utc=True)
    df['tx_hour'] = tx.dt.hour
    df['tx_dayofweek'] = tx.dt.dayofweek
    df['tx_month'] = tx.dt.month

    fpd_ts = pd.to_datetime(df['FIRST_PAYMENT_DUE_TIMESTAMP'], utc=True)
    df['days_to_first_due'] = (fpd_ts - tx).dt.total_seconds() / 86400

    mfsd = pd.to_datetime(df['MERCHANT_FIRST_SALE_DATE'], utc=True)
    df['merchant_tenure_days'] = (tx - mfsd).dt.days

    if rate_mask is None:
        rate_mask = df['FPD_15'].notna()
    global_mean = df.loc[rate_mask, 'FPD_15'].mean()
    m = 50
    for entity in ['MERCHANTID', 'CLERK_ID', 'ADMINID']:
        grp = df.loc[rate_mask].groupby(entity)['FPD_15'].agg(['mean', 'count'])
        grp['smoothed'] = (grp['count'] * grp['mean'] + m * global_mean) / (grp['count'] + m)
        df[f'{entity.lower()}_fpd_rate'] = df[entity].map(grp['smoothed'])
    state_grp = df.loc[rate_mask].groupby('STATE')['FPD_15'].agg(['mean', 'count'])
    state_grp['smoothed'] = (state_grp['count'] * state_grp['mean'] + m * global_mean) / (state_grp['count'] + m)
    df['state_fpd_rate'] = df['STATE'].map(state_grp['smoothed'])
    city_grp = df.loc[rate_mask].groupby('CITY')['FPD_15'].agg(['mean', 'count'])
    city_grp['smoothed'] = (city_grp['count'] * city_grp['mean'] + m * global_mean) / (city_grp['count'] + m)
    df['city_fpd_rate'] = df['CITY'].map(city_grp['smoothed'])
    model_grp = df.loc[rate_mask].groupby('MODEL')['FPD_15'].agg(['mean', 'count'])
    model_grp['smoothed'] = (model_grp['count'] * model_grp['mean'] + m * global_mean) / (model_grp['count'] + m)
    df['model_fpd_rate'] = df['MODEL'].map(model_grp['smoothed'])
    for entity in ['MERCHANTID', 'CLERK_ID', 'ADMINID']:
        counts = df.loc[rate_mask].groupby(entity)['FPD_15'].count()
        df[f'{entity.lower()}_order_count'] = df[entity].map(counts)
    return df

In [ ]:
def prep_data(orders_df, payments_df, selected_features, scale_data=True):
    """End-to-end data preparation: aggregate → engineer → impute → encode → scale.

    Parameters
    ----------
    orders_df : DataFrame — full orders table
    payments_df : DataFrame — payment history table
    selected_features : list[str] — columns to include as model inputs
    scale_data : bool — whether to z-score standardize

    Returns
    -------
    X_train : np.ndarray — Jan–Nov feature matrix
    X_test  : np.ndarray — Dec feature matrix
    y_train : np.ndarray — Jan–Nov labels
    scaler  : StandardScaler or None
    test_order_ids : pd.Series — FINANCEORDERID for each test row (for submission)
    """
    df = orders_df.copy()

    # Step 1 — Payment aggregations
    pay_agg = _aggregate_payments(payments_df)
    df = df.merge(pay_agg, on='FINANCEORDERID', how='left')

    # Step 2 — Feature engineering (rate_mask=None → all labelled rows)
    df = _engineer_features(df, rate_mask=None)

    # Step 3 — Train/test split on FPD_15 availability
    labelled = df['FPD_15'].notna()
    y_train = df.loc[labelled, 'FPD_15'].values
    test_order_ids = df.loc[~labelled, 'FINANCEORDERID']

    train_feat = df.loc[labelled, selected_features].copy()
    test_feat = df.loc[~labelled, selected_features].copy()

    # Step 4 — Imputation (training stats only → applied to both splits)
    cat_cols = train_feat.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = train_feat.select_dtypes(include='number').columns.tolist()

    medians = train_feat[num_cols].median()
    train_feat[num_cols] = train_feat[num_cols].fillna(medians)
    test_feat[num_cols] = test_feat[num_cols].fillna(medians)

    for col in cat_cols:
        fill = train_feat[col].mode().iloc[0] if not train_feat[col].mode().empty else 'UNKNOWN'
        train_feat[col] = train_feat[col].fillna(fill)
        test_feat[col] = test_feat[col].fillna(fill)

    # Step 5 — One-hot encoding with column alignment
    if cat_cols:
        train_feat = pd.get_dummies(train_feat, columns=cat_cols)
        test_feat = pd.get_dummies(test_feat, columns=cat_cols)
        train_feat, test_feat = train_feat.align(
            test_feat, join='left', axis=1, fill_value=0
        )

    # Step 6 — Scaling
    scaler = None
    if scale_data:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_feat.values.astype(np.float64))
        X_test = scaler.transform(test_feat.values.astype(np.float64))
    else:
        X_train = train_feat.values.astype(np.float64)
        X_test = test_feat.values.astype(np.float64)

    return X_train, X_test, y_train, scaler, test_order_ids

## Step 3 — Prepare Features

v4 features (pre-OHE): smoothed target encoding, CITY, MODEL, days_to_first_due.

In [ ]:
selected_features = [
    'FINANCE_AMOUNT', 'PURCHASE_AMOUNT', 'TOTAL_DUE', 'DOWN_PAYMENT_AMOUNT',
    'FACE_RECOGNITION_SCORE', 'IDVALIDATION_OVERALL_SCORE',
    'LIVENESS_SCORE', 'OVERALL_SCORE',
    'down_payment_ratio', 'finance_ratio',
    'tx_hour', 'tx_dayofweek', 'tx_month',
    'merchant_tenure_days', 'days_to_first_due',
    'merchantid_fpd_rate', 'clerk_id_fpd_rate', 'adminid_fpd_rate',
    'state_fpd_rate', 'city_fpd_rate', 'model_fpd_rate',
    'merchantid_order_count', 'clerk_id_order_count', 'adminid_order_count',
    'COUNTRY', 'LOCK_PRODUCT', 'MANUFACTURER',
    'LOCK_NAME', 'CURRENCY',
]

X_train, X_test, y_train, scaler, test_order_ids = prep_data(
    orders, payments, selected_features,
)
print(f'Training set:  {X_train.shape}  (Jan–Nov, all labelled orders)')
print(f'Test set:      {X_test.shape}  (Dec, unlabelled orders)')
print(f'Train FPD rate: {y_train.mean():.4f}')

## Step 4 — Train Winning Neural Network

FPDNet MLP: [128, 64, 32] hidden layers, lr=0.0005, relu. Nov AUC: 0.5661.

In [ ]:
class FPDNet(nn.Module):
    def __init__(self, input_dim, hidden_layers, activation='relu'):
        super().__init__()
        act_cls = nn.ReLU if activation == 'relu' else nn.LeakyReLU
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), act_cls(), nn.Dropout(0.4)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FPDNet(X_train.shape[1], [128, 64, 32], 'relu').to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
pos_weight = torch.tensor([(1 - y_train.mean()) / y_train.mean()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

tr_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
tr_loader = DataLoader(tr_ds, batch_size=1024, shuffle=True)

model.train()
for epoch in range(50):
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

model.eval()
print('Model trained on full Jan–Nov dataset.')
print(f'Features: {X_train.shape[1]}  Training samples: {X_train.shape[0]:,}')

## Step 5 — Predict December & Write Submission

Generate predicted probabilities for all December test orders, build the
submission DataFrame, validate it against the required format, and save.

In [ ]:
# Predict P(FPD_15 = 1) for each December order
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    dec_probs = model.predict_proba(X_test_t).cpu().numpy()

# Build submission DataFrame
submission = pd.DataFrame({
    'FINANCEORDERID': test_order_ids.astype(str).values,
    'FPD_15_pred': dec_probs,
})
submission = submission.sort_values('FINANCEORDERID').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'Mean predicted FPD probability: {submission["FPD_15_pred"].mean():.4f}')
print(f'Median predicted FPD probability: {submission["FPD_15_pred"].median():.4f}')
submission.head(10)

## Step 6 — Submission Validation

Verify the submission meets all Kaggle format requirements before uploading.

In [ ]:
print('Running submission validation checks...\n')

test_ids_set = set(test_ids['FINANCEORDERID'].astype(str))
sub_ids_set = set(submission['FINANCEORDERID'])

checks = [
    ('No duplicate FINANCEORDERIDs',
     not submission['FINANCEORDERID'].duplicated().any()),
    ('All test set IDs present',
     test_ids_set == sub_ids_set),
    ('All predictions in [0, 1]',
     (submission['FPD_15_pred'] >= 0).all() and (submission['FPD_15_pred'] <= 1).all()),
    ('No missing values',
     not submission.isnull().any().any()),
    ('Correct row count',
     len(submission) == len(test_ids)),
]

all_passed = True
for name, passed in checks:
    status = '[OK]' if passed else '[FAIL]'
    if not passed:
        all_passed = False
    print(f'  {status} {name}')

if all_passed:
    print('\nAll checks passed. submission.csv is ready for Kaggle upload.')
else:
    print('\nSome checks FAILED — fix before submitting!')